In [ ]:
import json


In [ ]:
# clean title field by removing empty strings

with open('recipes.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    
for recipe in data:
    recipe['title'] = [title for title in recipe['title'] if title.strip()]
    
with open('recipes.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

In [ ]:
with open('recipes.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
print(data[0]['title'])

In [ ]:
print(data[0]['ingredients'])

In [ ]:
with open('recipes.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
for recipe in data:
    print(recipe['title'])
    print(recipe['ingredients'])
    print(recipe['instructions'])
    print('---')
    

In [ ]:
# remove emojis and zero-width characters from text fields

import emoji
import re

def clean_text(text):
    if isinstance(text, list):
        return [clean_text(t) for t in text]

    if isinstance(text, str):
       
        text = emoji.replace_emoji(text, replace='')

        
        text = re.sub(r'[\u200d\u200c\u200b]', '', text)

        
        text = re.sub(r'\s+', ' ', text)

        return text.strip()

    return text

In [ ]:
text = "I love pizza! 🍕🔥 😂 🚀 🍕 🎯"

print(clean_text(text))

In [ ]:
with open('recipes.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
for recipe in data:
    recipe['title'] = clean_text(recipe['title'])
    print(recipe['title'])
    recipe['ingredients'] = clean_text(recipe['ingredients'])
    print(recipe['ingredients'])
    recipe['instructions'] = clean_text(recipe['instructions'])
    print(recipe['instructions'])
    print('---')

In [ ]:
# split ingredient into quantity and ingredient name


import re

units = {
    "كوب", "كوبين", "كاسات", "كاسة",
    "ملعقة", "ملاعق", "معلقة", "معلقتين",
    "ملعقة صغيرة", "معلقة صغيرة",
    "ملعقة كبيرة", "معلقة كبيرة",
    "حبة", "حبتين", "حبات",
    "ضمة", "ضمتين",
    "فص", "فصوص",
    "رشة", "بكيت", "علبة", "علبتين",
    "كيلو", "غرام", "جرام", "لتر",
    "نص", "ربع", "ثلث", "ونص"
}

default_units = {
    "ليمون": "حبة",
    "ليمونة": "حبة",
    "بصلة": "حبة",
    "بندورة": "حبة",
    "بطاطا": "حبة",
    "كوسا": "حبة",
    "سن": "حبة",
    "ثوم": "فص"
}

def split_ingredient(item):
    item = item.strip()
    words = item.split()

    quantity = []
    ingredient = []

    i = 0
    while i < len(words):
        word = words[i]

        # رقم أو كسر
        if re.match(r'^[\d\.\/½]+$', word):
            quantity.append(word)

            # إذا الكلمة اللي بعدها وحدة
            if i + 1 < len(words):
                next_word = words[i + 1]

                # وحدة مكونة من كلمتين
                if i + 2 < len(words):
                    two_words = next_word + " " + words[i + 2]
                    if two_words in units:
                        quantity.append(two_words)
                        i += 3
                        continue

                if next_word in units:
                    quantity.append(next_word)
                    i += 2
                    continue

            i += 1
            continue

        # كوب ونص / معلقة صغيرة
        if i + 1 < len(words):
            two_words = word + " " + words[i + 1]
            if two_words in units:
                quantity.append(two_words)
                i += 2
                continue

        # وحدة لحالها
        if word in units and len(quantity) == 0:
            quantity.append(word)
            i += 1
            continue

        ingredient.extend(words[i:])
        break

    ingredient_text = " ".join(ingredient)

    # إذا في رقم بدون وحدة نحاول نضيف وحدة افتراضية
    if len(quantity) == 1 and re.match(r'^[\d\.\/½]+$', quantity[0]) and ingredient:
        first_word = ingredient[0]
        if first_word in default_units:
            quantity.append(default_units[first_word])

    return {
        "quantity": " ".join(quantity),
        "ingredient": ingredient_text
    }

In [ ]:
# remove generic phrases from ingredient names

ignore_patterns = [
    r'اي نوع',
    r'حسب الرغبة',
    r'حسب الرغبه',
    r'متوفر',
    r'انا استخدمت',
    r'للتزيين',
    r'للتزين',
    r'مكونات',
    r'على الوجه',
    r'على الوجة',
    r'للوجه',
    r'للوجة',
    r'اختياري',
    r'اختياريًا',
    r'او حسب',
    r'أو حسب',
    r'اذا توفر',
    r'إذا توفر',
    r'يفضل',
    r'يمكن',
    r'حسب الذوق',
    r'حسب الحاجة',
    r'حسب الحاجه',
    r'نوع حسب',
    r'اي شي',
    r'أي شي',
    r'اي شيء',
    r'أي شيء',
    r'للحشو',
    r'للتقديم',
    r'للتجميل',
    r'للتزيين',
    r'حسب المتوفر'
]

def is_valid_ingredient(text):
    text = text.strip().lower()

    # إذا النص قصير جدًا نخليه
    if len(text) <= 2:
        return True

    # إذا النص كله عبارة عن كلام عام وغير مفيد
    for pattern in ignore_patterns:
        if re.search(pattern, text):
            return False

    return True

In [ ]:
# for recipe in data:
#     print(recipe['title'])

#     recipe['ingredients'] = [
#         split_ingredient(item)
#         for item in recipe['ingredients']
#     ]

#     recipe['ingredients'] = [
#         ingredient
#         for ingredient in recipe['ingredients']
#         if is_valid_ingredient(ingredient['ingredient'])
#     ]

#     print(recipe['ingredients'])
#     print(recipe['instructions'])
#     print('---')

In [ ]:
# with open('recipes.json', 'w', encoding='utf-8') as f:
#     json.dump(data, f, ensure_ascii=False, indent=4)